# 06 Modeling — Random Forest for Future Realized Volatility

This notebook trains a **Random Forest** to predict future realized volatility: given what we know about an asset today, predict how volatile it will be over the next 21 trading days (about one month).

This is one half of the team's model comparison — another team member is training a **linear regression** on the same task. To make the comparison fair, both models must use the **same target, the same train/test split dates, and the same evaluation metrics** (R² and MAE on the held-out test set). Only the model differs.

Why a Random Forest as the contrast to linear regression?
- Linear regression assumes each feature has one constant, additive effect. A forest of decision trees can learn **nonlinear effects** (e.g., volatility mattering differently when it is extreme) and **interactions** (e.g., momentum meaning something different for bonds than for stocks) automatically.
- The performance gap between the two models is itself a finding: it measures how much nonlinearity actually exists in the data.

The model is trained as one pooled model on all 21 tickers together. Each row is one (date, ticker) observation, and ticker-specific behavior comes through the features (each ticker carries its own rolling volatility, momentum, and sector dummies).

# Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import TimeSeriesSplit, GridSearchCV
from sklearn.metrics import mean_absolute_error, r2_score

# Paths

In [ ]:
INTEGRATED_PATH = Path("../data/processed/integrated")
MODELING_PATH = Path("../data/processed/modeling")

MODELING_PATH.mkdir(parents=True, exist_ok=True)

# Load Integrated Data

We start from the modeling base dataset created in notebook 04 (long format: one row per date per ticker).

In [ ]:
df = pd.read_csv(
    INTEGRATED_PATH / "modeling_base_dataset.csv",
    parse_dates=["Date"]
)

df = df.sort_values(["ticker", "Date"]).reset_index(drop=True)

print(df.shape)
df.head()

# Build Features

All rolling features are computed **within each ticker** using `groupby("ticker")`, so one asset's history never bleeds into another's. Every feature only uses information available up to and including day *t* — no peeking at the future.

These mirror the features explored in notebook 05:
- Rolling mean returns (5d, 21d, 63d) — recent momentum
- Rolling volatility (5d, 21d, 63d) — recent risk level
- Price relative to its moving average (21d, 63d) — trend position
- Macro context: risk-free rate and monthly CPI change (from FRED)

**NOTE:** the teammate building the linear regression model should use this exact same feature list so the model comparison is fair.

In [ ]:
g = df.groupby("ticker")

for window in [5, 21, 63]:
    df[f"ret_mean_{window}d"] = g["daily_return"].transform(lambda s: s.rolling(window).mean())
    df[f"vol_{window}d"] = g["daily_return"].transform(lambda s: s.rolling(window).std())

for window in [21, 63]:
    moving_average = g["adjusted_close"].transform(lambda s: s.rolling(window).mean())
    df[f"price_to_ma_{window}d"] = df["adjusted_close"] / moving_average

# Build Target — Forward 21-Day Realized Volatility

The target for a row on day *t* is the standard deviation of that ticker's daily returns over days *t+1* to *t+21*. We compute a 21-day rolling std (which at row *t* covers days *t-20 … t*) and then shift it back 21 rows within each ticker, so the value that lands on day *t* covers exactly the next 21 trading days.

In [ ]:
df["fwd_vol_21d"] = (
    df.groupby("ticker")["daily_return"]
    .transform(lambda s: s.rolling(21).std().shift(-21))
)

# Encode Categorical Metadata

One-hot encode sector and asset type. This is how the pooled model tells asset classes apart — bonds, gold, and tech stocks each get their own baseline volatility level.

In [ ]:
df = pd.get_dummies(df, columns=["gics_sector", "asset_type"], drop_first=False)

dummy_columns = [c for c in df.columns if c.startswith(("gics_sector_", "asset_type_"))]
print(f"{len(dummy_columns)} dummy columns created")

# Assemble Model Dataset

Drop the warm-up rows (first 63 days per ticker have incomplete rolling features) and the last 21 days per ticker (no future window to compute the target from).

In [ ]:
FEATURE_COLUMNS = (
    ["ret_mean_5d", "ret_mean_21d", "ret_mean_63d",
     "vol_5d", "vol_21d", "vol_63d",
     "price_to_ma_21d", "price_to_ma_63d",
     "risk_free_rate_decimal", "cpi_pct_change"]
    + dummy_columns
)
TARGET_COLUMN = "fwd_vol_21d"

model_df = df.dropna(subset=FEATURE_COLUMNS + [TARGET_COLUMN]).copy()

print("Rows before dropna:", len(df))
print("Rows after dropna:", len(model_df))
print("Features:", len(FEATURE_COLUMNS))

# Train/Test Split — By Date, Not By Row

We split chronologically: everything before 2024 is training data, 2024 onward is the held-out test set. A random shuffled split would leak the future into training (the model would see 2025 market conditions while being "tested" on 2024) and give fake-good scores.

All 21 tickers appear on both sides of the split — the split is on **time**, not on assets.

**NOTE:** `SPLIT_DATE` must match the linear regression notebook exactly, otherwise the two models are being tested on different data and the comparison means nothing.

In [ ]:
SPLIT_DATE = "2024-01-01"

# TimeSeriesSplit later assumes rows are in chronological order, so sort by Date
model_df = model_df.sort_values("Date").reset_index(drop=True)

train_df = model_df[model_df["Date"] < SPLIT_DATE]
test_df = model_df[model_df["Date"] >= SPLIT_DATE]

X_train, y_train = train_df[FEATURE_COLUMNS], train_df[TARGET_COLUMN]
X_test, y_test = test_df[FEATURE_COLUMNS], test_df[TARGET_COLUMN]

print(f"Train: {X_train.shape[0]} rows, {train_df['Date'].min().date()} to {train_df['Date'].max().date()}")
print(f"Test:  {X_test.shape[0]} rows, {test_df['Date'].min().date()} to {test_df['Date'].max().date()}")

# Baseline — Persistence

Before training anything, we need a baseline to beat. The simplest credible forecast of future volatility is **current volatility**: predict that the next 21 days will look like the last 21 days. Any model that can't beat this isn't adding value. Both team models should report this same baseline.

In [ ]:
baseline_pred = test_df["vol_21d"]

baseline_r2 = r2_score(y_test, baseline_pred)
baseline_mae = mean_absolute_error(y_test, baseline_pred)

print(f"Persistence baseline — R2: {baseline_r2:.4f}, MAE: {baseline_mae:.6f}")

# How a Random Forest Works (Briefly)

A **decision tree** predicts by asking a sequence of yes/no questions about the features ("is `vol_21d` > 0.012? → is `asset_type_ETF` true? → …") until it reaches a leaf, then predicts the average target of the training rows that landed in that leaf. A single deep tree memorizes the training data (overfits); a single shallow tree is too crude (underfits).

A **random forest** trains hundreds of trees, each on a random resample of the rows and a random subset of features at each split, then averages their predictions. The randomness makes the trees disagree in different ways, and averaging cancels out their individual overfitting.

Unlike linear/Ridge regression, no feature scaling is needed — trees only ask "greater or less than a threshold?", so the units of a feature don't matter.

The two hyperparameters we tune control tree complexity:
- **`max_depth`** — how many questions deep a tree may go. Deeper = more complex patterns, more overfitting risk.
- **`min_samples_leaf`** — how many training rows a leaf must contain. Larger = smoother, more conservative predictions.

In [ ]:
param_grid = {
    "max_depth": [4, 8, 16, None],
    "min_samples_leaf": [1, 20, 100],
}

grid_search = GridSearchCV(
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
    param_grid,
    cv=TimeSeriesSplit(n_splits=5),
    scoring="neg_mean_squared_error",
)

grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)

# Validation Curves — What Tuning Actually Does

Cross-validated error for every hyperparameter combination, using `TimeSeriesSplit` so validation folds always come **after** the data the fold trained on. Each line is one `min_samples_leaf` setting, plotted against `max_depth`. Look for the classic pattern: error falls as the trees get more expressive, then flattens or rises once they start overfitting.

In [ ]:
cv_results = pd.DataFrame(grid_search.cv_results_)
cv_results["mse"] = -cv_results["mean_test_score"]
cv_results["depth_label"] = cv_results["param_max_depth"].astype(str)

fig, ax = plt.subplots(figsize=(8, 5))
for leaf_size, group in cv_results.groupby("param_min_samples_leaf"):
    ax.plot(group["depth_label"], group["mse"], marker="o", label=f"min_samples_leaf={leaf_size}")
ax.set_xlabel("max_depth")
ax.set_ylabel("Cross-validated MSE")
ax.set_title("Random Forest Validation Curves")
ax.legend()
plt.tight_layout()
plt.show()

# Evaluate on the Held-Out Test Set

The tuned model now sees the 2024+ data for the first time. These are the headline numbers to put next to the linear regression model's results (same split, same metrics).

In [ ]:
y_pred = grid_search.predict(X_test)

rf_r2 = r2_score(y_test, y_pred)
rf_mae = mean_absolute_error(y_test, y_pred)

comparison = pd.DataFrame({
    "model": ["Persistence baseline", "Random Forest (tuned)", "Linear Regression (teammate)"],
    "test_R2": [baseline_r2, rf_r2, np.nan],
    "test_MAE": [baseline_mae, rf_mae, np.nan],
})

comparison

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, s=4, alpha=0.3)
limit = max(y_test.max(), y_pred.max())
ax.plot([0, limit], [0, limit], color="red", linestyle="--", label="perfect prediction")
ax.set_xlabel("Actual forward 21d volatility")
ax.set_ylabel("Predicted forward 21d volatility")
ax.set_title("Predicted vs Actual (test set)")
ax.legend()
plt.tight_layout()
plt.show()

# Which Features Matter?

Random forests report **feature importances**: how much each feature reduced prediction error across all the splits in all the trees (importances sum to 1). Expect the recent volatility features (`vol_21d`, `vol_63d`) to dominate — volatility clustering is the main signal.

In [ ]:
importances = (
    pd.Series(grid_search.best_estimator_.feature_importances_, index=FEATURE_COLUMNS)
    .sort_values()
)

fig, ax = plt.subplots(figsize=(8, 8))
importances.plot.barh(ax=ax)
ax.set_xlabel("Feature importance")
ax.set_title("Random Forest Feature Importances")
plt.tight_layout()
plt.show()

# Sanity Check — Does the Pooled Model Distinguish Tickers?

One pooled model, but per-ticker inputs. If it were just predicting "average volatility," every ticker's predictions would look the same. Instead, quiet assets (AGG) should sit near zero and volatile ones (AAPL, AMZN) much higher.

In [ ]:
test_results = test_df.copy()
test_results["predicted_fwd_vol_21d"] = y_pred

per_ticker = (
    test_results.groupby("ticker")[[TARGET_COLUMN, "predicted_fwd_vol_21d"]]
    .mean()
    .sort_values(TARGET_COLUMN)
)

fig, ax = plt.subplots(figsize=(9, 6))
per_ticker.plot.bar(ax=ax)
ax.set_ylabel("Mean 21d volatility (test period)")
ax.set_title("Actual vs Predicted Volatility by Ticker")
plt.tight_layout()
plt.show()

# Save Test Predictions

Saved per-row so the two models' predictions can be compared side by side later (and fed into the portfolio/dashboard work).

In [ ]:
output_columns = ["Date", "ticker", TARGET_COLUMN, "predicted_fwd_vol_21d"]

test_results[output_columns].to_csv(
    MODELING_PATH / "random_forest_volatility_test_predictions.csv",
    index=False
)

print("Saved:", MODELING_PATH / "random_forest_volatility_test_predictions.csv")

# Final Summary

In this notebook we:

- Built leakage-safe features per ticker (rolling returns, rolling volatility, price-to-moving-average, macro context) from the integrated long-format dataset
- Defined the target as forward 21-day realized volatility
- Split train/test **by date** (train < 2024, test ≥ 2024) to avoid look-ahead bias
- Established a persistence baseline (future volatility = current volatility)
- Trained one pooled Random Forest across all 21 tickers and tuned `max_depth` / `min_samples_leaf` with `GridSearchCV` + `TimeSeriesSplit`
- Verified the pooled model distinguishes quiet assets from volatile ones

**For the model comparison:** the linear regression notebook must use the same feature list, target, `SPLIT_DATE`, and metrics; then the comparison table above can be filled in with its test R² and MAE. The gap between the two models measures how much nonlinearity the Random Forest found beyond the linear relationships.